# R/S benchmark — 4. NN training (global R, S, t $\to$ lambda)

Trains one `MLPRegressor` for `lambda 1` and one for `lambda 2` on the dataset stacked by
[`03_generate_dataset_nn.ipynb`](03_generate_dataset_nn.ipynb). Unlike the PCE stage
(`02_train_pce.ipynb`, one model per time step), this fits a single global model that also takes
$t$ as an input — query it with any $(R, S, t)$ triple, no need to pick a PCE for a specific time
step first.

`lambda 3` / `lambda 4` are **not** modelled here — read them back from the emulator dataset
directly, as scoped when this pipeline was set up.

Functions come from [`functions.py`](../functions.py): `train_and_validate_nn_lambda_benchmark`.
Prediction plots and the KL-divergence check are in
[`04_train_nn_plot.ipynb`](04_train_nn_plot.ipynb).

## 1. Libraries

In [5]:
%matplotlib inline
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import dill
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rcParams.update({
                        'font.family': 'serif',
                        'mathtext.fontset': 'cm',
                        'axes.unicode_minus': False
                    })
from sklearn.model_selection import train_test_split

from functions import *

## 2. Config

`n_latent_samples` must match [`03_generate_dataset_nn.ipynb`](03_generate_dataset_nn.ipynb) — it
names the file being loaded.

In [6]:
n_latent_samples = 2500   # must match stage 3 — it is the filename prefix

feature_cols = ['r', 's', 'Time (years)']
target_cols  = ['lambda 1', 'lambda 2']

test_frac          = 0.2
hidden_layer_sizes = (64, 64)
max_iter           = 500
n_iter_no_change   = 15
random_state       = 42

fig_size   = (5, 4)      # size of each individual figure, in inches
fig_format = 'png'       # format each figure is saved in ('pdf', 'png', ...)
fig_dpi    = 300         # resolution the figure is saved at (dots per inch)

label_fontsize = 14   # font size of the axis labels
tick_fontsize  = 12   # font size of the tick numbers

xlim = None   # e.g. (-5, 5) to fix the axis; None = auto-scaled to the data, per lambda
ylim = None   # e.g. (-5, 5) to fix the axis; None = auto-scaled to the data, per lambda

## 3. Load the stacked dataset

In [7]:
with open(f'{n_latent_samples}_dataset_nn_benchmark.pkl', 'rb') as f:
    df_nn = dill.load(f)

print(f"Loaded {len(df_nn)} rows")
df_nn.head()

Loaded 25000 rows


,r,s,Time (years),lambda 1,lambda 2,lambda 3,lambda 4
0,5.206907,1.854493,0.0,3.354403,6.357913,0.142157,0.131525
1,5.286790,2.431395,0.0,2.858020,5.292953,0.142157,0.131525
2,5.269668,2.067211,0.0,3.204697,5.916525,0.142157,0.131525
3,5.194718,2.683213,0.0,2.514461,4.938048,0.142157,0.131525
4,5.579349,0.870576,0.0,4.708886,8.263479,0.142157,0.131525


## 4. Train and validate

In [8]:
print("="*60)
print("TRAINING THE BENCHMARK NN")
print("="*60)

result = train_and_validate_nn_lambda_benchmark(
                                                   df_nn=df_nn,
                                                   feature_cols=feature_cols,
                                                   target_cols=target_cols,
                                                   test_frac=test_frac,
                                                   hidden_layer_sizes=hidden_layer_sizes,
                                                   max_iter=max_iter,
                                                   n_iter_no_change=n_iter_no_change,
                                                   random_state=random_state,
                                                   n_latent_samples=n_latent_samples,
                                                   output_dir='.',
                                                )

result['statistics']

TRAINING THE BENCHMARK NN

----------------------------------------
TRAINING NN LAMBDA MODELS
----------------------------------------
  20000 train rows, 5000 val rows
  lambda 1: R² = 0.999890, MSE = 0.00016, iterations = 23
  lambda 2: R² = 0.999454, MSE = 0.00205, iterations = 64
The NN models, scaler and validation stats have been saved!


,MSE lambda 1,R² lambda 1,MSE lambda 2,R² lambda 2
0,0.000164,0.99989,0.00205,0.999454
